# Understanding AI Coding Agents

## Learning Objective

By the end of this lesson, you will be able to describe how a simple AI coding agent works. Including:

1. Calls an inference API.
2. What a system prompt is and how it shapes behaviour.
3. Handling tool calls.
4. The core agent loop.

## Calling An Inference API

It's just another REST API.

In this example we're using the native Google Gemini REST API. 

Many provides offer an OpenAI compatible REST API which is fairly similar.

To trigger inference the client makes a HTTP(S) POST request sending the input as a JSON payload.

## Python Setup

Imports some modules for HTTPS requests and handling JSON

In [ ]:
import json
import os

import requests
from IPython.display import display, Markdown

Set the URL, Headers and API Key

In [ ]:
URL = "https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent" 

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

headers = {
    "x-goog-api-key": GEMINI_API_KEY,
    "Content-Type": "application/json",
}

## Some Utility Functions

In [ ]:
# def print_answer(resp, debug=False):
#     """ Prints just the answer from the payload """
#     res = json.loads(resp.text)
    
#     if debug:
#         print(f"{res}\n")
    
#     role = res['candidates'][0]['content']['role']
#     answer = res['candidates'][0]['content']['parts'][0]['text']
#     display(Markdown(f"{role}: {answer}"))

In [ ]:

def parse_response(resp, debug=False):
    res = json.loads(resp.text)
    role = res['candidates'][0]['content']['role']
    answer = res['candidates'][0]['content']['parts'][0]['text']
    return role, answer

def make_request(payload):
    response = requests.post(URL, headers=headers, data=json.dumps(payload))
    if response.status_code == 200:
        return parse_response(response)
    else:
        print(f"Errror, status code: {response.status_code} details: {response.text}")



    
# def make_request_with_tools(payload, debug=False):
#     response = requests.post(URL, headers=headers, data=json.dumps(payload))
#     if response.status_code == 200:
#         res = json.loads(response.text)
#         if debug:
#             print(res)
#             print()
#             parts = res['candidates'][0]['content']['parts'][0]
#             role = res['candidates'][0]['content']['role']
#             if 'functionCall' in parts:
#                 print(f"Tool call: {parts['functionCall']}")
#     else:
#         print(f"Errror, status code: {response.status_code} details: {response.text}")

## A Simple Inference Request

A simple call to an LLM for a response involves sending a JSON payload in a HTTPS POST request. 

The content of the "chat" is in the 'text' section of the 'parts'. The 'contents' array is an array of objects that make up the contents of the call to the LLM.

In [ ]:
payload = {
    "contents": [
      {
        "role": "user",
        "parts": [
          {
            "text": "Explain how AI works in a few words"
          }
        ]
      }
    ]
}

role, response = make_request(payload)
display(Markdown(f"{role.title()}:\n{response}"))

## Defining Behaviour With A System Prompt

A system prompt (known as system instruction for Gemini) defines the behaviour of the model. It is given more weight than the user prompts.

The system prompt is defined as text in the payload we send to the API for inference.

In [ ]:
system_instructions = {
    "system_instruction": {
      "parts": [
        {
          "text": "You are a coding agent, act like a software engineer. Your name is John's AI Coding Bot."
        }
      ]
    }
}

In [ ]:
payload = system_instructions | {
    "contents": [
      {
        "role": "user",
        "parts": [
          {
            "text": "Who are you? What do you do?"
          }
        ]
      }
    ]
}

role, response = make_request(payload)
display(Markdown(f"{role.title()}:\n{response}"))

## Tool Calls

Tools are local functions our agent loop can run. We tell the model the tools exist when we run inference.

To do that we use the tools section of the JSON payload:

In [ ]:
tools = {
"tools": [
        {
            "function_declarations": [
                {
                    "name": "write_text",
                    "description": "writes some text to a file",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "filename": {
                                "type": "string",
                                "description": "The filename to write the text to"
                            },
                            "text": {
                                "type": "string",
                                "description": "The text to write to the file"
                            }
                        },
                        "required": [
                            "filename",
                            "text"
                        ]
                    }
                }
            ]
        }
    ]
}

In [ ]:
payload = system_instructions | tools | {    
    "contents": [
        {
            "role": "user",
            "parts": [
                {
                    "text": "Who are you? What do you do? What tools can you use?"
                }
            ]
        }
    ]
}

role, response = make_request(payload)
display(Markdown(f"{role.title()}:\n{response}"))

## Tool Calls And Multiturn Work

For the model to be able to use tools we need to support a multi-turn workflow.

1. Take user prompt.
2. Run inference.
3. If there are tool call requests, run the tools.
4. Return the output of the tool runs to the model and run inference.
5. If there are more tool calls repeat 3, otherwise provide output to user.

### Tools Are Just Functions

Let's provide the write_text tool we told the LLM we have.

In [ ]:
def write_text(filename, code):
    """ A simple tool to write text (code) to a file """
    try:
        path = os.path.join("./", filename)
        with open(path, "w") as f:
            f.write(code)
    except Exception as e:
        return False

    return True

## Ask The Agent To Write Code

In [ ]:
payload = system_instructions | tools | {    
    "contents": []
}

first_user_request = {
    "role": "user",
    "parts": [
        {
            "text": "Write a python script to output the first 10 numbers of the Fibonacci sequence."
        }
    ]
}

payload["contents"].append(first_user_request)

response = requests.post(URL, headers=headers, data=json.dumps(payload))
if response.status_code == 200:
    res = json.loads(response.text)
    print(response.text)
else:
    print(f"Errror, status code: {response.status_code} details: {response.text}")

In [ ]:
tool_call = {}
parts = res['candidates'][0]['content']['parts'][0]
role = res['candidates'][0]['content']['role']
if 'functionCall' in parts:
    tool_call = res['candidates'][0]['content']
    function = parts['functionCall']['name']
    filename = parts['functionCall']['args']['filename']
    text = parts['functionCall']['args']['text']
    print(f"Tool call requested\nFunction: `{function}` with filename: `{filename}`")
    print("Provided text:")
    print(text)
if 'text' in parts:
    print(f"{role}: {parts['text']}")

## Making The Tool Call:

In [ ]:
if function == 'write_text':
    tool_call_result = write_code(filename, text)
    print(f"Writing code returned: {tool_call_result}")

In [ ]:
!cat $filename

## Creating A Response

Having called that function with the provide arguments, we then send the model a tool response based on the function output. That response would look like the following:

In [ ]:
tool_response = {
    "role": "user",
    "parts": [{
        "functionResponse": {
            "name": "write_code",
            "response": {
                "success": tool_call_result,
                "output": "File written successfully"
            }
        }
    }]
}

In [ ]:
# Next we call the model, recording it's tool call
payload["contents"].append(tool_call)

# And the tool response as new messages in the contents list
payload["contents"].append(tool_response)

# Let's see the payload
print(json.dumps(payload, indent=4))


In [ ]:
role, response = make_request(payload)
display(Markdown(f"{role.title()}:\n{response}"))

## What is a Coding Agent?

An agent that helps the user write code and build software. 

It does that by "chatting" to the user, writing code to files and then building, running and testing the code using tools.

To keep running the agent runs in a loop.

### The Agent Loop



In [ ]:
def parse_response():
    """ Updated parser to handle tool calls """
    pass

In [ ]:
def append_user_prompt(payload, user_prompt):
    return payload["contents"].append({
        "role": "user",
        "parts": [
            {
                "text": user_prompt
            }
        ]
    })

def append_model_response(payload, response):
    payload["contents"].append({
        "role": "model",
        "parts": [
            {
                "text": response
            }
        ]
    })

In [ ]:
while True:
    print("\r---\n")
    req = input("You  : ")
    if req.lower() in {"quit", "exit"}:
        break

    if req.lower() in {"/clear", "/new"}:
        payload = system_instructions | {    
            "contents": []
        }
        continue

    append_user_prompt(payload, req)

    role, response = make_request(payload)

    # TODO handle tool calls

    display(Markdown(f"{role.title()}:\n{response}"))

    append_model_response(payload, response)